# Create Datasets

## Create Matches Dataset

In [161]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display


MATCHES_JSON_PATH = Path.cwd().parent / "data" / "matches.json"

matches_json = json.loads(MATCHES_JSON_PATH.read_text())

#Update matches_json so that home_team and away_team are not dictionaries of team data
for match in matches_json:
    home_team = match.pop("home_team")
    away_team = match.pop("away_team")

    match["home_team_id"] = home_team["id"]
    match["away_team_id"] = away_team["id"]

    match["home_team_name"] = home_team["name"]
    match["away_team_name"] = away_team["name"]

#Create a DataFrame from the updated matches_json
matches_df = pd.DataFrame(matches_json)
matches_df.head()

,id,date_time,home_score,away_score,competition_id,competition_name,season_id,competition_edition_id,season,status,home_team_id,away_team_id,home_team_name,away_team_name
0,114243,2025-10-11T19:00:00Z,83,68,8,Liga ACB,51,211,2025-2026,closed,1330,1319,BAXI Manresa,Coviran Granada
1,114234,2025-10-18T19:00:00Z,84,88,8,Liga ACB,51,211,2025-2026,closed,1331,1322,Casademont Zaragoza,Leche Rio Breogan
2,114169,2025-12-14T11:30:00Z,80,77,8,Liga ACB,51,211,2025-2026,closed,1334,1336,CB UCAM Murcia,Club Joventut Badalona
3,114099,2026-01-31T20:00:00Z,77,71,8,Liga ACB,51,211,2025-2026,closed,1315,1316,Basquet Girona,San Pablo Burgos
4,114086,2026-02-08T17:00:00Z,75,97,8,Liga ACB,51,211,2025-2026,closed,1328,1326,Dreamland Gran Canaria,Bitci Baskonia


In [162]:
matches_df['id'].value_counts()

id
114243    1
114234    1
114169    1
114099    1
114086    1
178442    1
179612    1
184439    1
188630    1
191313    1
Name: count, dtype: int64

## Create Individual Game Datasets - Game Summaries and Rosters

### 

In [163]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

#Create Roster DataFrame for a given team key
def roster_frame(game_data,team_key):
    team = game_data[team_key]
    roster = pd.DataFrame(team.get('players', []))
    roster['teamId'] = team.get('teamId')
    roster['teamName'] = team.get('teamName')
    return roster

#Create Game Dataframe and Roster DataFrames for a given game_id
def create_game_data(game_id):
    GAME_ID = str(game_id)
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next((p for p in candidates if (p / 'data' / 'matches' / GAME_ID).is_dir()), None)
    if REPO_ROOT is None:
        raise FileNotFoundError('Could not locate data/matches/' + GAME_ID + ' from ' + str(Path.cwd()))

    GAME_DIR = REPO_ROOT / 'data' / 'matches' / GAME_ID
    GAME_DATA_PATH = GAME_DIR / f'{GAME_ID}_game_data.json'

    
    with GAME_DATA_PATH.open(encoding='utf-8') as f:
        game_data = json.load(f)
    
    game_summary = pd.DataFrame([{
        key: game_data.get(key)
        for key in ('gameId', 'date', 'competitionName', 'seasonName', 'competitionId', 'seasonId')
    }])

    home_roster = roster_frame(game_data, 'homeTeam')
    away_roster = roster_frame(game_data, 'awayTeam')

    home_roster['home_away'] = 'home'
    away_roster['home_away'] = 'away'
    
    return game_summary, home_roster, away_roster

In [164]:
#Create List of Game DataFrames for all games in matches_df
game_summaries = []
home_rosters = []
away_rosters = []

for game_id in matches_df['id']:
    game_summary, home_roster, away_roster = create_game_data(game_id)
    game_summaries.append(game_summary)
    home_rosters.append(home_roster)
    away_rosters.append(away_roster)

In [165]:
game_summaries[0]

,gameId,date,competitionName,seasonName,competitionId,seasonId
0,114243,2025-10-11T19:00:00Z,Liga ACB,2025-2026,8,51


In [166]:
home_rosters[0]

,playerId,firstName,lastName,jersey,teamId,teamName,home_away
0,15630,Kaodirichi,Akobundu-Ehiogu,0,1330,BAXI Manresa,home
1,18503,Grant,Golden,33,1330,BAXI Manresa,home
2,18534,Alfonso,Plummer,99,1330,BAXI Manresa,home
3,35806,Retin,Obasohan,32,1330,BAXI Manresa,home
4,59179,Louis,Olinde,19,1330,BAXI Manresa,home
5,59180,Alex,Reyes,8,1330,BAXI Manresa,home
6,59181,Marcis,Steinbergs,12,1330,BAXI Manresa,home
7,59183,Hugo,Benitez,2,1330,BAXI Manresa,home
8,59184,Agustin,Ubal,11,1330,BAXI Manresa,home
9,59186,Max,Gaspa,4,1330,BAXI Manresa,home


In [167]:
away_rosters[0]

,playerId,firstName,lastName,jersey,teamId,teamName,home_away
0,25271,Elias,Valtonen,11,1319,Coviran Granada,away
1,25326,Ivan,Aurrecoechea,15,1319,Coviran Granada,away
2,28247,Zach,Hankins,35,1319,Coviran Granada,away
3,28648,Travis,Munnings,22,1319,Coviran Granada,away
4,35028,Matt,Thomas,13,1319,Coviran Granada,away
5,59217,Luka,Bozic,29,1319,Coviran Granada,away
6,59219,Beka,Burjanadze,77,1319,Coviran Granada,away
7,59220,Jonathan,Rousselle,2,1319,Coviran Granada,away
8,59221,Jovan,Kljajic,23,1319,Coviran Granada,away
9,59232,Eduardo,Duran,16,1319,Coviran Granada,away


### Check Rosters

In [168]:
sum = 0
for idx in range(len(home_rosters)):
    print(idx, home_rosters[idx].shape)
    sum += home_rosters[idx].shape[0]
print("Total number of rows in home rosters:", sum)

sum = 0
for idx in range(len(away_rosters)):
    print(idx, away_rosters[idx].shape)
    sum += away_rosters[idx].shape[0]
print("Total number of rows in away rosters:", sum)

0 (12, 7)
1 (11, 7)
2 (12, 7)
3 (11, 7)
4 (12, 7)
5 (12, 7)
6 (12, 7)
7 (12, 7)
8 (12, 7)
9 (12, 7)
Total number of rows in home rosters: 118
0 (12, 7)
1 (11, 7)
2 (10, 7)
3 (12, 7)
4 (11, 7)
5 (12, 7)
6 (12, 7)
7 (12, 7)
8 (12, 7)
9 (12, 7)
Total number of rows in away rosters: 116


In [169]:
temp = pd.DataFrame()

temp = pd.concat(home_rosters+away_rosters, ignore_index=True)


In [170]:
player_aliases_df = pd.read_csv(Path.cwd().parent / "data" / "player_id_aliases.csv")
player_aliases_df

,player_id,player_name,canonical_player_id,canonical_player_name,acb_player_id
0,59129,Nihad Djedovic,59129,Nihad Djedovic,20202039
1,59237,Nihad Dedovic,59129,Nihad Djedovic,20202039
2,92815,Alejandro REYES,59180,Alex Reyes,20204133
3,59180,Alex Reyes,59180,Alex Reyes,20204133
4,32852,Ruben Lopez,32852,Ruben Lopez,30000132
5,59094,De La Torre Ruben Lopez,32852,Ruben Lopez,30000132
6,27976,Christ Koumadje,27976,Christ Koumadje,30000550
7,94602,Khalifa KOUMADJE,27976,Christ Koumadje,30000550
8,11247,Kameron Milton Allante Taylor,11247,Kameron Milton Allante Taylor,30001981
9,446,Kameron Taylor,11247,Kameron Milton Allante Taylor,30001981


In [171]:
temp.loc[temp['playerId'].isin(list(player_aliases_df['player_id']))]

,playerId,firstName,lastName,jersey,teamId,teamName,home_away
5,59180,Alex,Reyes,8,1330,BAXI Manresa,home
33,59094,De La Torre,Ruben Lopez,16,1334,CB UCAM Murcia,home
39,58436,Derek Ryan,Needham,5,1315,Basquet Girona,home
40,59187,Jose,Ignacio Vildoza,10,1315,Basquet Girona,home
81,59237,Nihad,Dedovic,14,1339,Malaga,home
186,446,Kameron,Taylor,1,1325,Valencia BC,away


## Create Dynamic Event Datasets for Each Game

In [172]:
#Create Game Dataframe and Roster DataFrames for a given game_id
def create_dynamic_events(game_id):
    GAME_ID = str(game_id)
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next((p for p in candidates if (p / 'data' / 'matches' / GAME_ID).is_dir()), None)
    if REPO_ROOT is None:
        raise FileNotFoundError('Could not locate data/matches/' + GAME_ID + ' from ' + str(Path.cwd()))

    GAME_DIR = REPO_ROOT / 'data' / 'matches' / GAME_ID
    EVENTS_PATH = GAME_DIR / f'{GAME_ID}_dynamic_events.json'

    
    with EVENTS_PATH.open(encoding='utf-8') as f:
        events_json = json.load(f)
    
    event_frames = {name: pd.DataFrame(records) for name, records in events_json.items()}
    
    return event_frames #This returns a dictionary of DataFrames for each game, where the keys are the event types and the values are the corresponding DataFrames.

In [173]:
#Create a List of Dynamic Event Dictionaries 
#Each Item in the List is a Dictionary of DataFrames for a given game_id, where the keys are the event types and the values are the corresponding DataFrames.
dynamic_event_frames = {}

for game_id in matches_df['id']:
    event_frames = create_dynamic_events(game_id)
    dynamic_event_frames[game_id] = event_frames


In [174]:
#Keys in Each Dictionary of DataFrames for a given game_id
list(dynamic_event_frames.keys())

[114243,
 114234,
 114169,
 114099,
 114086,
 178442,
 179612,
 184439,
 188630,
 191313]

In [175]:
list(dynamic_event_frames[114243].keys())

['chances',
 'possessions',
 'shots',
 'rebounds',
 'turnovers',
 'free_throws',
 'fouls',
 'timeouts',
 'passes',
 'touches',
 'dribbles',
 'picks',
 'isolations',
 'handoffs',
 'drives',
 'posts',
 'off_ball_screens',
 'closeouts',
 'chance_players',
 'matchups']

In [176]:
dynamic_event_frames[114243]['chances'].columns

Index(['id', 'possessionId', 'gameId', 'season', 'period', 'startFrame',
       'endFrame', 'startWallClock', 'endWallClock', 'startGameClock',
       'endGameClock', 'offTeamId', 'defTeamId', 'startType', 'outcome',
       'offPlayerIds', 'defPlayerIds', 'homeStartScore', 'awayStartScore',
       'ptsScored', 'startShotClock', 'endShotClock', 'ballInPaint',
       'endOfPossession', 'endLoc', 'frontcourtFrame', 'transition',
       'lastChanceId', 'qualityIndex', 'usable'],
      dtype='str')

In [177]:
dynamic_event_frames[114243]['chances'][['offPlayerIds', 'defPlayerIds', 'qualityIndex', 'endLoc']]

,offPlayerIds,defPlayerIds,qualityIndex,endLoc
0,"[25271, 35028, 59217, 59233, 28247]","[59181, 35806, 18503, 59214, 59180]",4,"[-40.19028871391076, 2.5695640583989503]"
1,"[25271, 35028, 59217, 59233, 28247]","[59181, 35806, 18503, 59214, 59180]",5,"[-35.50996555118112, 5.179497826443555]"
2,"[59181, 35806, 18503, 59214, 59180]","[25271, 35028, 59217, 59233, 28247]",5,"[1.9922131315616267, -1.8023793778707882]"
3,"[25271, 35028, 59217, 59233, 28247]","[59181, 35806, 18503, 59214, 59180]",2,"[-23.529773622047244, -6.657798146325459]"
4,"[59181, 35806, 18503, 59214, 59180]","[25271, 35028, 59217, 59233, 28247]",5,"[-33.83878772965876, -11.612379019028907]"
...,...,...,...,...
211,"[25271, 35028, 59220, 28247, 59219]","[59181, 35806, 18503, 59227, 18534]",1,"[-40.03649934383202, -2.9924848261154855]"
212,"[59181, 35806, 18503, 59227, 18534]","[25271, 35028, 59220, 28247, 59219]",1,"[-36.69927001312384, -2.1210117290026536]"
213,"[25271, 35028, 59220, 28247, 59219]","[59181, 35806, 18503, 59227, 18534]",4,"[-39.487983923884514, 9.7169250328084]"
214,"[59183, 35806, 59186, 59180, 18534]","[25271, 59220, 28648, 59217, 59219]",3,"[-31.972810039369936, -1.2780537442579774]"


In [178]:
dynamic_event_frames[114243]['passes'].columns

Index(['id', 'gameId', 'season', 'period', 'chanceId', 'possessionId',
       'startFrame', 'endFrame', 'startWallClock', 'endWallClock',
       'startGameClock', 'endGameClock', 'shotClock', 'offTeamId', 'defTeamId',
       'passerId', 'receiverId', 'toReceiverId', 'complete', 'turnover',
       'ledToShot', 'assistOpp', 'inbounds', 'backcourt', 'passerLoc',
       'receiverLoc', 'passerRegion', 'receiverRegion', 'distance', 'touchId'],
      dtype='str')

## Construct Single - Chance Level Data Frame per Shot Attempt

### Shots

In [179]:
dynamic_event_frames[114243]['shots'].columns

Index(['id', 'gameId', 'season', 'period', 'chanceId', 'possessionId',
       'startFrame', 'startWallClock', 'startGameClock', 'endFrame',
       'endWallClock', 'endGameClock', 'location', 'region', 'shotClock',
       'shooterId', 'offTeamId', 'defTeamId', 'three', 'outcome', 'fouled',
       'foulId', 'assisted', 'assistOpp', 'passerId', 'blocked', 'blockerId',
       'distance', 'dribblesBefore', 'shotType', 'complexShotType',
       'catchAndShoot', 'shotQuality', 'createdFromPaint', 'contested',
       'contestLevel', 'releaseTime', 'closestDefDist', 'closestDefId',
       'touchId'],
      dtype='str')

In [180]:
shot_fields = ['id', 'chanceId', 'possessionId', 'startGameClock', 'endGameClock', 'shotClock', #Shot, Chance and Time ID and fields
                                        'shooterId', #Shooter Information
                                        'region', 'location', 'distance', 'releaseTime', #Shot Location, distance, and Time Information
                                        'dribblesBefore', #Dribble Context Information
                                        'assisted', 'assistOpp', 'passerId', 'createdFromPaint', #Shot Assist Information
                                        'shotType', 'complexShotType', 'catchAndShoot', #Shot Type Information
                                        'closestDefDist', 'closestDefId', 'contested', 'contestLevel', 'blocked', 'blockerId', #Shot Defense Information
                                        'three', 'outcome', #Shot Outcome Information
                                        'fouled', 'foulId', #Shot Aftermath Information - Fouls
                                        'offTeamId', 'defTeamId', 'touchId']

len(shot_fields)
                                    

32

In [181]:
shots_df = pd.DataFrame()
for game_id in matches_df['id']:
    X = dynamic_event_frames[game_id]['shots'][shot_fields].copy()
    X['game_id'] = game_id
    shots_df = pd.concat([shots_df, X[['game_id'] + shot_fields]], ignore_index=True)

shots_df.rename(columns={'id': 'shot_id'}, inplace=True)
shots_df.shape


(1459, 33)

In [182]:
shots_df

,game_id,shot_id,chanceId,possessionId,startGameClock,endGameClock,shotClock,shooterId,region,location,...,contestLevel,blocked,blockerId,three,outcome,fouled,foulId,offTeamId,defTeamId,touchId
0,114243,shot-114243-1-0,chance-114243-1-3,possession-114243-1-2,556.535461,555.455261,13.88,35028,middle two,"[-23.529773622047244, -6.657798146325459]",...,open,False,NaN,False,False,False,NaN,1319,1330,touch-114243-1-16
1,114243,shot-114243-1-1,chance-114243-1-4,possession-114243-1-3,546.053833,544.933655,16.20,59180,right wing two,"[-33.83878772965876, -11.612379019028907]",...,average,False,NaN,False,False,False,NaN,1330,1319,touch-114243-1-19
2,114243,shot-114243-1-2,chance-114243-1-5,possession-114243-1-4,533.331909,532.131714,13.40,25271,NaN,None,...,NaN,False,NaN,True,False,False,NaN,1319,1330,touch-114243-1-22
3,114243,shot-114243-1-3,chance-114243-1-8,possession-114243-1-5,516.852051,515.572754,14.00,59181,right wing three,"[-26.246719160104988, -20.261749507874015]",...,average,False,NaN,True,True,False,NaN,1330,1319,touch-114243-1-31
4,114243,shot-114243-1-4,chance-114243-1-10,possession-114243-1-7,486.125122,486.084991,16.84,35806,right wing two,"[-23.273053907985055, -13.599702277650838]",...,open,False,NaN,False,True,True,foul-114243-1-3,1330,1319,touch-114243-1-39
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1454,191313,shot-191313-4-136,chance-191313-4-195,possession-191313-4-146,38.720000,38.320000,20.92,31739,ra,"[-38.180774278215225, -1.3118553662237533]",...,average,False,NaN,False,True,False,NaN,1331,1322,touch-191313-4-601
1455,191313,shot-191313-4-137,chance-191313-4-197,possession-191313-4-148,20.600000,19.319999,20.48,59207,right wing three,"[-22.63687967746086, -12.800438931489749]",...,average,False,NaN,True,False,False,NaN,1331,1322,touch-191313-4-609
1456,191313,shot-191313-4-138,chance-191313-4-198,possession-191313-4-148,15.520000,14.200000,10.80,31739,left corner three,"[-42.92773950131233, 21.80765912073491]",...,plus,False,NaN,True,False,True,foul-191313-4-45,1331,1322,touch-191313-4-610
1457,191313,shot-191313-4-139,chance-191313-4-200,possession-191313-4-150,7.900000,6.420000,0.00,31739,right wing three,"[-17.03217273622047, -17.224409448818896]",...,plus,False,NaN,True,True,False,NaN,1331,1322,touch-191313-4-615


In [183]:
list(dynamic_event_frames[114243].keys())

['chances',
 'possessions',
 'shots',
 'rebounds',
 'turnovers',
 'free_throws',
 'fouls',
 'timeouts',
 'passes',
 'touches',
 'dribbles',
 'picks',
 'isolations',
 'handoffs',
 'drives',
 'posts',
 'off_ball_screens',
 'closeouts',
 'chance_players',
 'matchups']

### Passes

In [184]:
dynamic_event_frames[114243]['passes'].columns

Index(['id', 'gameId', 'season', 'period', 'chanceId', 'possessionId',
       'startFrame', 'endFrame', 'startWallClock', 'endWallClock',
       'startGameClock', 'endGameClock', 'shotClock', 'offTeamId', 'defTeamId',
       'passerId', 'receiverId', 'toReceiverId', 'complete', 'turnover',
       'ledToShot', 'assistOpp', 'inbounds', 'backcourt', 'passerLoc',
       'receiverLoc', 'passerRegion', 'receiverRegion', 'distance', 'touchId'],
      dtype='str')

In [185]:
pass_fields = ['gameId', 'id', 'chanceId', 'possessionId', 'startGameClock', 'endGameClock', 'shotClock', #Shot, Chance and Time ID and fields
                                        'passerId', 'passerLoc', 'passerRegion', #Passer Information
                                        'receiverId', 'receiverLoc', 'receiverRegion', #Receiver Information
                                        'toReceiverId', #Defender Information - Is Not Null if the pass was intercepted by a defender
                                        'inbounds', 'backcourt', 'distance', #Pass Distance Information
                                        'complete', 'turnover', #Pass Completion Information
                                        'ledToShot', 'assistOpp', #Pass Outcome Information
                                        'offTeamId', 'defTeamId', 'touchId']

In [186]:
set(dynamic_event_frames[114243]['passes'].columns) - set(pass_fields)

{'endFrame',
 'endWallClock',
 'period',
 'season',
 'startFrame',
 'startWallClock'}

In [187]:
passes_df = pd.concat([dynamic_event_frames[game_id]['passes'][pass_fields] for game_id in matches_df['id']], ignore_index=True)

passes_df.shape

(4786, 24)

In [188]:
passes_df.head()

,gameId,id,chanceId,possessionId,startGameClock,endGameClock,shotClock,passerId,passerLoc,passerRegion,...,inbounds,backcourt,distance,complete,turnover,ledToShot,assistOpp,offTeamId,defTeamId,touchId
0,114243,pass-114243-1-0,chance-114243-1-0,possession-114243-1-0,597.837891,597.837891,20.88,59217,"[11.707215797244094, 6.183614255249344]",backcourt,...,False,True,0.000000,True,False,False,False,1319,1330,touch-114243-1-0
1,114243,pass-114243-1-1,chance-114243-1-0,possession-114243-1-0,593.033264,592.632874,16.08,59233,"[-0.13386483580421588, 1.9399888246391075]",far,...,False,False,15.381129,True,False,False,False,1319,1330,touch-114243-1-2
2,114243,pass-114243-1-2,chance-114243-1-0,possession-114243-1-0,590.350647,589.830139,13.40,25271,"[-23.224758038057733, 18.626455872703414]",left wing three,...,False,False,22.423036,True,False,False,False,1319,1330,touch-114243-1-3
3,114243,pass-114243-1-3,chance-114243-1-0,possession-114243-1-0,588.028442,NaN,11.08,59217,"[-41.42060367454068, 2.3805312910104988]",ra,...,False,False,NaN,None,False,False,False,1319,1330,touch-114243-1-4
4,114243,pass-114243-1-4,chance-114243-1-1,possession-114243-1-0,587.588013,587.588013,NaN,59233,"[-46.34186351706037, 14.11017470472441]",left corner two,...,True,False,24.576070,True,False,False,False,1319,1330,touch-114243-1-5


In [189]:
list(dynamic_event_frames[114243].keys())

['chances',
 'possessions',
 'shots',
 'rebounds',
 'turnovers',
 'free_throws',
 'fouls',
 'timeouts',
 'passes',
 'touches',
 'dribbles',
 'picks',
 'isolations',
 'handoffs',
 'drives',
 'posts',
 'off_ball_screens',
 'closeouts',
 'chance_players',
 'matchups']

### Touches

In [190]:
size = 0
for idx in range(len(matches_df['id'])):
    game_id = matches_df['id'][idx]
    size += dynamic_event_frames[game_id]['touches'].shape[0]

size

6662

In [191]:
dynamic_event_frames[game_id]['touches'].columns

Index(['id', 'gameId', 'season', 'period', 'chanceId', 'possessionId',
       'startFrame', 'endFrame', 'startWallClock', 'endWallClock',
       'startGameClock', 'endGameClock', 'shotClock', 'offTeamId', 'defTeamId',
       'playerId', 'defenderId', 'closestDefLoc', 'location', 'endLoc',
       'numDribbles', 'ledToShot', 'touchTime', 'outcomes', 'regionsIn',
       'endClosestDefLoc', 'direct'],
      dtype='str')

In [192]:
touch_fields = [
    'id', 'chanceId', 'possessionId', 'startGameClock', 'endGameClock', 'shotClock',
    'playerId', 'defenderId', 'closestDefLoc', 'location', 'endLoc', 'numDribbles',
    'ledToShot', 'touchTime', 'outcomes', 'regionsIn', 'endClosestDefLoc', 'direct',
    'offTeamId', 'defTeamId'
]

touches_df = pd.concat([
    dynamic_event_frames[game_id]['touches'][touch_fields].assign(
        gameId=(dynamic_event_frames[game_id]['touches']['gameId']
                if 'gameId' in dynamic_event_frames[game_id]['touches'].columns
                else game_id)
    )[['gameId'] + touch_fields]
    for game_id in matches_df['id']
], ignore_index=True)

touches_df.shape

(6662, 21)

In [193]:
touches_df

,gameId,id,chanceId,possessionId,startGameClock,endGameClock,shotClock,playerId,defenderId,closestDefLoc,...,endLoc,numDribbles,ledToShot,touchTime,outcomes,regionsIn,endClosestDefLoc,direct,offTeamId,defTeamId
0,114243,touch-114243-1-0,chance-114243-1-0,possession-114243-1-0,600.000000,597.837891,24.00,59217,35806.0,None,...,"[11.707215797244094, 6.183614255249344]",0,False,3.12,[PASS],[backcourt],"[5.760693487532809, 0.3612448224245407]",False,1319,1330
1,114243,touch-114243-1-1,chance-114243-1-0,possession-114243-1-0,597.837891,597.757812,20.88,59217,35806.0,"[5.760693487532809, 0.3612448224245407]",...,"[11.543173802493444, 5.8375881725721905]",0,False,0.08,[],[backcourt],"[5.209614911417343, 0.4365375348589212]",False,1319,1330
2,114243,touch-114243-1-2,chance-114243-1-0,possession-114243-1-0,597.757812,593.033264,20.80,59233,59214.0,"[5.209614911417343, 0.4365375348589212]",...,"[-0.13386483580421588, 1.9399888246391075]",3,False,4.72,[PASS],"[backcourt, far]","[-4.42144438976378, -0.5811155624589895]",False,1319,1330
3,114243,touch-114243-1-3,chance-114243-1-0,possession-114243-1-0,592.632874,590.350647,15.68,25271,59180.0,"[-24.503772965879264, 6.481581159776903]",...,"[-23.224758038057733, 18.626455872703414]",0,False,2.28,"[ASTF, PASS]","[middle three, left wing three]","[-24.070599573490806, 16.61950459317586]",True,1319,1330
4,114243,touch-114243-1-4,chance-114243-1-0,possession-114243-1-0,589.830139,588.028442,12.88,59217,59181.0,"[-35.44844980314961, 7.817626312335958]",...,"[-41.42060367454068, 2.3805312910104988]",2,False,1.80,"[FOUL, FOU_N, PASS]","[key, left corner two, ra]","[-40.03649934383202, 3.3417148540026247]",False,1319,1330
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6657,191313,touch-191313-4-614,chance-191313-4-200,possession-191313-4-150,11.220000,8.700000,0.00,59207,59107.0,"[10.039882709973217, -2.6221087598426367]",...,"[-15.173884514435695, -12.501794209317586]",2,False,2.52,"[AST, AST3, PASS]","[backcourt, far, right wing three]","[-18.057435203412073, -12.988793881233596]",True,1331,1322
6658,191313,touch-191313-4-615,chance-191313-4-200,possession-191313-4-150,8.620000,7.900000,0.00,31739,59107.0,"[-16.21452591863517, -7.821471046587927]",...,"[-17.03217273622047, -17.224409448818896]",0,True,0.72,"[FGM, FGM3, TMO]","[far, right wing three]","[-20.133591699475065, -15.28922654199475]",True,1331,1322
6659,191313,touch-191313-4-616,chance-191313-4-201,possession-191313-4-151,5.200000,5.100000,NaN,59104,59207.0,"[-14.72533218503937, -22.132853510498688]",...,"[-16.82712024278215, -27.195086942257216]",0,False,1.20,[],[far],"[-18.60851377952756, -21.722748523622048]",False,1322,1331
6660,191313,touch-191313-4-618,chance-191313-4-204,possession-191313-4-152,4.000000,2.100000,NaN,31739,59107.0,"[-30.732242454068242, -1.2870247908464567]",...,"[-16.41701525590551, 2.950833538385827]",1,False,2.32,[],"[key, middle two, middle three]","[-20.73593339895013, 5.302529322506562]",False,1331,1322


### Dribbles

In [194]:
dribble_fields = [
    'id', 'chanceId', 'possessionId', 'gameClock', 'ballhandlerId', 'offTeamId',
    'defTeamId', 'location', 'region', 'touchId'
]

dribbles_df = pd.concat([
    dynamic_event_frames[game_id]['dribbles'][dribble_fields].assign(
        gameId=(dynamic_event_frames[game_id]['dribbles']['gameId']
                if 'gameId' in dynamic_event_frames[game_id]['dribbles'].columns
                else game_id)
    )[['gameId'] + dribble_fields]
    for game_id in matches_df['id']
], ignore_index=True)

dribbles_df.shape

(13962, 11)

### Picks

In [195]:
pick_fields = [
    'id', 'chanceId', 'possessionId', 'gameClock', 'direct', 'location', 'region',
    'shotClock', 'ballhandlerId', 'screenerId', 'ballhandlerDefId', 'screenerDefId',
    'offTeamId', 'defTeamId', 'locationType', 'bhrDefType', 'scrDefType',
    'bhrOutcomes', 'scrOutcomes', 'handoffPick', 'touchId'
]

picks_df = pd.concat([
    dynamic_event_frames[game_id]['picks'][pick_fields].assign(
        gameId=(dynamic_event_frames[game_id]['picks']['gameId']
                if 'gameId' in dynamic_event_frames[game_id]['picks'].columns
                else game_id)
    )[['gameId'] + pick_fields]
    for game_id in matches_df['id']
], ignore_index=True)

picks_df.shape

(1520, 22)

### Isolations

In [196]:
isolation_fields = [
    'id', 'chanceId', 'possessionId', 'startGameClock', 'endGameClock', 'direct',
    'location', 'region', 'endLoc', 'shotClock', 'ballhandlerId', 'ballhandlerDefId',
    'offTeamId', 'defTeamId', 'bhrOutcomes', 'touchId'
]

isolations_df = pd.concat([
    dynamic_event_frames[game_id]['isolations'][isolation_fields].assign(
        gameId=(dynamic_event_frames[game_id]['isolations']['gameId']
                if 'gameId' in dynamic_event_frames[game_id]['isolations'].columns
                else game_id)
    )[['gameId'] + isolation_fields]
    for game_id in matches_df['id']
], ignore_index=True)

isolations_df.shape

(180, 17)

### Handoffs

In [197]:
handoff_fields = [
    'id', 'chanceId', 'possessionId', 'gameClock', 'direct', 'location', 'region',
    'shotClock', 'receiverId', 'setterId', 'receiverDefId', 'setterDefId', 'offTeamId',
    'defTeamId', 'locationType', 'receiverDefType', 'setterDefType', 'bhrOutcomes',
    'handoffPick', 'touchId'
]

handoffs_df = pd.concat([
    dynamic_event_frames[game_id]['handoffs'][handoff_fields].assign(
        gameId=(dynamic_event_frames[game_id]['handoffs']['gameId']
                if 'gameId' in dynamic_event_frames[game_id]['handoffs'].columns
                else game_id)
    )[['gameId'] + handoff_fields]
    for game_id in matches_df['id']
], ignore_index=True)

handoffs_df.shape

(417, 21)

### Drives

In [198]:
drive_fields = [
    'id', 'chanceId', 'possessionId', 'startGameClock', 'endGameClock', 'direct',
    'location', 'region', 'endLoc', 'shotClock', 'ballhandlerId', 'ballhandlerDefId',
    'offTeamId', 'defTeamId', 'category', 'blowby', 'endType', 'direction',
    'bhrOutcomes', 'dribbleThrough', 'touchId'
]

drives_df = pd.concat([
    dynamic_event_frames[game_id]['drives'][drive_fields].assign(
        gameId=(dynamic_event_frames[game_id]['drives']['gameId']
                if 'gameId' in dynamic_event_frames[game_id]['drives'].columns
                else game_id)
    )[['gameId'] + drive_fields]
    for game_id in matches_df['id']
], ignore_index=True)

drives_df.shape

(660, 22)

### Posts

In [199]:
post_fields = [
    'id', 'chanceId', 'possessionId', 'startGameClock', 'endGameClock', 'direct',
    'location', 'region', 'endLoc', 'shotClock', 'ballhandlerId', 'ballhandlerDefId',
    'offTeamId', 'defTeamId', 'bhrOutcomes', 'section', 'dribbledInto', 'touchId'
]

posts_df = pd.concat([
    dynamic_event_frames[game_id]['posts'][post_fields].assign(
        gameId=(dynamic_event_frames[game_id]['posts']['gameId']
                if 'gameId' in dynamic_event_frames[game_id]['posts'].columns
                else game_id)
    )[['gameId'] + post_fields]
    for game_id in matches_df['id']
], ignore_index=True)

posts_df.shape

(105, 19)

### Off Ball Screens

In [200]:
off_ball_screen_fields = [
    'id', 'chanceId', 'possessionId', 'gameClock', 'direct', 'location', 'region',
    'shotClock', 'cutterId', 'screenerId', 'cutterDefId', 'screenerDefId', 'offTeamId',
    'defTeamId', 'cutterDefType', 'screenerDefType', 'outcomes', 'ledToTouch',
    'ledToShot', 'scrOutcomes', 'touchId', 'scrTouchId'
]

off_ball_screens_df = pd.concat([
    dynamic_event_frames[game_id]['off_ball_screens'][off_ball_screen_fields].assign(
        gameId=(dynamic_event_frames[game_id]['off_ball_screens']['gameId']
                if 'gameId' in dynamic_event_frames[game_id]['off_ball_screens'].columns
                else game_id)
    )[['gameId'] + off_ball_screen_fields]
    for game_id in matches_df['id']
], ignore_index=True)

off_ball_screens_df.shape

(1103, 23)

### Closeouts

In [201]:
closeout_fields = [
    'id', 'chanceId', 'possessionId', 'touchGameClock', 'touchFrame', 'touchWallClock',
    'startGameClock', 'endGameClock', 'direct', 'location', 'region', 'startLoc',
    'endLoc', 'touchLoc', 'startDistance', 'touchDistance', 'endDistance',
    'ballhandlerId', 'ballhandlerDefId', 'offTeamId', 'defTeamId', 'bhrOutcomes',
    'bhrAction', 'touchId'
]

closeouts_df = pd.concat([
    dynamic_event_frames[game_id]['closeouts'][closeout_fields].assign(
        gameId=(dynamic_event_frames[game_id]['closeouts']['gameId']
                if 'gameId' in dynamic_event_frames[game_id]['closeouts'].columns
                else game_id)
    )[['gameId'] + closeout_fields]
    for game_id in matches_df['id']
], ignore_index=True)

closeouts_df.shape

(601, 25)

### Chance Players

In [202]:
chance_player_fields = [
    'id', 'chanceId', 'possessionId', 'playerId', 'offense', 'startMatchupId',
    'endMatchupId', 'shotLoc', 'rimLoc', 'reboundLoc', 'minutesPlayedBefore'
]

chance_players_df = pd.concat([
    dynamic_event_frames[game_id]['chance_players'][chance_player_fields].assign(
        gameId=(dynamic_event_frames[game_id]['chance_players']['gameId']
                if 'gameId' in dynamic_event_frames[game_id]['chance_players'].columns
                else game_id)
    )[['gameId'] + chance_player_fields]
    for game_id in matches_df['id']
], ignore_index=True)

chance_players_df.shape

(20540, 12)

### Matchups

In [203]:
matchup_fields = [
    'id', 'chanceId', 'possessionId', 'startGameClock', 'endGameClock', 'shotClock',
    'defPlayerId', 'offPlayerId'
]

matchups_df = pd.concat([
    dynamic_event_frames[game_id]['matchups'][matchup_fields].assign(
        gameId=(dynamic_event_frames[game_id]['matchups']['gameId']
                if 'gameId' in dynamic_event_frames[game_id]['matchups'].columns
                else game_id)
    )[['gameId'] + matchup_fields]
    for game_id in matches_df['id']
], ignore_index=True)

matchups_df.shape

(14416, 9)

## Unified Chance-Level Action Timeline

`actions_df` stacks touches, dribbles, passes, shots, and the play-action events into one event-level table across every chance, including chances that end in a turnover, foul, out of bounds or end of period. `chance_has_shot` (last column) flags chances that contain a shot attempt; filter on it, plus `frames_before_shot >= 0` to drop events after the release, to get the shot timeline. Each row is one event, ordered by frame within `gameId` and `chanceId`; event-specific source columns are retained, so unrelated fields are null on a given row.

Player columns per row:
- `actor_id`: who performed the action (passer, dribbler, cutter, driver, ...). Handoffs mirror picks, so the receiver is the actor and the setter is the secondary actor.
- `secondary_actor_id`: pass receiver, pick / off-ball screener, or handoff setter.
- `defender_id`: the actor's defender (closest defender at release for shots; the touch's assigned defender for dribbles and passes).
- `player_with_ball_id`: who held the ball at the action frame. For off-ball screens this is the touch in progress at the screen, not the cutter.

Chance-level columns repeated on every row: `chance_shot_id`, `chance_shooter_id`, `shot_frame`, `chance_shot_result` (`make` / `miss`; all null when the chance has no shot), plus `transition`, `usable`, `qualityIndex`, `startType`, `offPlayerIds`, `defPlayerIds` from `chances`. `frames_before_shot`, `seconds_before_shot` and `in_lookback` (within `LOOKBACK_SECONDS` of release) support the 2-3 second lookback. Touch details are joined by `touchId`; `chance_players` and `matchups` are not separate rows because they describe chance-level context rather than actions.

In [204]:
# Stack event-level rows into one timeline. Player-lineup and matchup tables are
# contextual rather than actions, so they are not added as separate timeline rows.
action_event_types = [
    'touches', 'dribbles', 'passes', 'shots', 'picks', 'isolations', 'handoffs',
    'drives', 'posts', 'off_ball_screens', 'closeouts'
]

# Primary actor, secondary actor and the primary actor's defender per action type.
# Handoffs mirror picks: the receiver is the ball handler (bhrOutcomes describe the
# receiver) and the setter plays the screener's role.
action_actor_fields = {
    'touches': 'playerId',
    'dribbles': 'ballhandlerId',
    'passes': 'passerId',
    'shots': 'shooterId',
    'picks': 'ballhandlerId',
    'isolations': 'ballhandlerId',
    'handoffs': 'receiverId',
    'drives': 'ballhandlerId',
    'posts': 'ballhandlerId',
    'off_ball_screens': 'cutterId',
    'closeouts': 'ballhandlerId',
}
action_secondary_fields = {
    'passes': 'receiverId',
    'picks': 'screenerId',
    'handoffs': 'setterId',
    'off_ball_screens': 'screenerId',
}
action_defender_fields = {
    'touches': 'defenderId',
    'shots': 'closestDefId',
    'picks': 'ballhandlerDefId',
    'isolations': 'ballhandlerDefId',
    'handoffs': 'receiverDefId',
    'drives': 'ballhandlerDefId',
    'posts': 'ballhandlerDefId',
    'off_ball_screens': 'cutterDefId',
    'closeouts': 'ballhandlerDefId',
}

FPS = 25
LOOKBACK_SECONDS = 3

action_rows = []
for game_id in matches_df['id']:
    for action_type in action_event_types:
        event_df = dynamic_event_frames[game_id][action_type].copy()
        if event_df.empty:
            continue

        if 'gameId' not in event_df.columns:
            event_df['gameId'] = game_id
        event_df['action_type'] = action_type
        event_df['action_id'] = event_df['id']
        if 'touchId' not in event_df.columns:
            event_df['touchId'] = pd.NA
        if action_type == 'touches':
            event_df['touchId'] = event_df['id']

        event_df['actor_id'] = event_df[action_actor_fields[action_type]]
        event_df['secondary_actor_id'] = event_df.get(action_secondary_fields.get(action_type), pd.NA)
        event_df['defender_id'] = event_df.get(action_defender_fields.get(action_type), pd.NA)
        if action_type in ('passes', 'handoffs') and 'receiverId' in event_df.columns:
            event_df['receiver_id'] = event_df['receiverId']
        else:
            event_df['receiver_id'] = pd.NA

        action_rows.append(event_df)

actions_df = pd.concat(action_rows, ignore_index=True, sort=False)

# Add the ballhandler and summarized touch context to each event linked to a touch.
# touchId is a key on its own; some events point at a touch logged under the previous chance.
touch_context_fields = [
    'playerId', 'defenderId', 'numDribbles', 'touchTime', 'outcomes',
    'regionsIn', 'location', 'endLoc', 'startFrame', 'endFrame',
    'startGameClock', 'endGameClock', 'shotClock'
]
touch_context_rows = []
for game_id in matches_df['id']:
    touch_context = dynamic_event_frames[game_id]['touches'].copy()
    if touch_context.empty:
        continue
    if 'gameId' not in touch_context.columns:
        touch_context['gameId'] = game_id
    available_fields = [
        field for field in touch_context_fields if field in touch_context.columns
    ]
    touch_context = touch_context[['gameId', 'id'] + available_fields]
    touch_context = touch_context.rename(columns={'id': 'touchId'})
    touch_context = touch_context.rename(columns={
        field: f'touch_{field}' for field in available_fields
    })
    touch_context_rows.append(touch_context)

touch_context_df = pd.concat(touch_context_rows, ignore_index=True, sort=False)
actions_df = actions_df.merge(
    touch_context_df,
    on=['gameId', 'touchId'],
    how='left',
    validate='many_to_one'
)

# Dribbles and passes carry no defender of their own, so use the touch's assigned defender.
no_defender_rows = actions_df['action_type'].isin(['dribbles', 'passes'])
actions_df.loc[no_defender_rows, 'defender_id'] = actions_df.loc[no_defender_rows, 'touch_defenderId']
for id_column in ('actor_id', 'secondary_actor_id', 'defender_id', 'receiver_id'):
    actions_df[id_column] = pd.to_numeric(actions_df[id_column], errors='coerce')

# Normalize event timing and add the shot result: make, miss, or missing for non-shots.
for frame_column in ('startFrame', 'endFrame', 'frame', 'touchFrame'):
    if frame_column not in actions_df.columns:
        actions_df[frame_column] = pd.NA

actions_df['action_frame'] = (
    actions_df['startFrame']
    .combine_first(actions_df['frame'])
    .combine_first(actions_df['touchFrame'])
).astype(float)
actions_df['shot'] = pd.NA
shot_rows = actions_df['action_type'].eq('shots')
actions_df.loc[shot_rows, 'shot'] = actions_df.loc[shot_rows, 'outcome'].map({
    True: 'make',
    False: 'miss',
})

# Player holding the ball at the action frame. Most events happen within the ball
# handler's touch, with two exceptions: a handoff links to the receiver's touch after
# the exchange (the setter has the ball at the handoff), and an off-ball screen links
# to the cutter's later touch, so use whichever touch is in progress at the screen.
actions_df['player_with_ball_id'] = actions_df['touch_playerId'].combine_first(
    actions_df['actor_id']
)
handoff_rows = actions_df['action_type'].eq('handoffs')
actions_df.loc[handoff_rows, 'player_with_ball_id'] = actions_df.loc[handoff_rows, 'setterId']

screen_rows = actions_df['action_type'].eq('off_ball_screens')
touch_spans = (
    touch_context_df[['gameId', 'touch_startFrame', 'touch_endFrame', 'touch_playerId']]
    .dropna(subset=['touch_startFrame'])
    .astype({'touch_startFrame': float})
    .sort_values('touch_startFrame')
)
screens = (
    actions_df.loc[screen_rows, ['gameId', 'action_frame']]
    .dropna()
    .sort_values('action_frame')
    .reset_index()
)
in_progress = pd.merge_asof(
    screens, touch_spans,
    left_on='action_frame', right_on='touch_startFrame',
    by='gameId', direction='backward'
).set_index('index')
in_progress = in_progress[in_progress['action_frame'] <= in_progress['touch_endFrame']]
actions_df.loc[screen_rows, 'player_with_ball_id'] = (
    in_progress['touch_playerId'].reindex(actions_df.index[screen_rows])
)

# Keep the event frame and identifiers, but omit redundant season/clock metadata.
actions_df = actions_df.drop(
    columns=['id', 'season', 'period', 'startWallClock', 'endWallClock'],
    errors='ignore'
)

# Shot context for chances that contain a shot attempt (null for the rest). One chance
# holds two shots (a putback), so key off the last shot. Events after the release have
# negative frames_before_shot.
shot_context = (
    actions_df.loc[actions_df['action_type'].eq('shots')]
    .sort_values('action_frame')
    .groupby(['gameId', 'chanceId'])
    .agg(
        chance_shot_id=('action_id', 'last'),
        chance_shooter_id=('actor_id', 'last'),
        shot_frame=('action_frame', 'last'),
        chance_shot_result=('shot', 'last'),
    )
)
actions_df = actions_df.join(shot_context, on=['gameId', 'chanceId'], how='left')
actions_df['frames_before_shot'] = actions_df['shot_frame'] - actions_df['action_frame']
actions_df['seconds_before_shot'] = actions_df['frames_before_shot'] / FPS
actions_df['in_lookback'] = actions_df['seconds_before_shot'].between(0, LOOKBACK_SECONDS)

# Chance-level context: transition (Stage 1), tracking quality, and on-court lineups.
chance_context_fields = [
    'transition', 'usable', 'qualityIndex', 'startType', 'offPlayerIds', 'defPlayerIds'
]
chance_context_df = pd.concat([
    dynamic_event_frames[game_id]['chances'][['gameId', 'id'] + chance_context_fields]
    for game_id in matches_df['id']
], ignore_index=True).rename(columns={'id': 'chanceId'})
actions_df = actions_df.merge(
    chance_context_df,
    on=['gameId', 'chanceId'],
    how='left',
    validate='many_to_one'
)

actions_df = actions_df.sort_values(
    ['gameId', 'chanceId', 'action_frame', 'endFrame'],
    kind='stable',
    na_position='last'
).reset_index(drop=True)
actions_df['sequence_in_chance'] = (
    actions_df.groupby(['gameId', 'chanceId'], dropna=False).cumcount() + 1
)
actions_df['chance_has_shot'] = actions_df['shot_frame'].notna()

actions_df.shape

(31455, 133)

In [205]:
actions_df[['gameId', 'chanceId', 'sequence_in_chance', 'action_type', 'seconds_before_shot', 'in_lookback', 'actor_id', 'secondary_actor_id', 'player_with_ball_id', 'defender_id', 'chance_shooter_id', 'chance_shot_result']].head(20)

,gameId,chanceId,sequence_in_chance,action_type,seconds_before_shot,in_lookback,actor_id,secondary_actor_id,player_with_ball_id,defender_id,chance_shooter_id,chance_shot_result
0,114086,chance-114086-1-0,1,touches,NaN,False,24268.0,NaN,24268.0,15347.0,NaN,NaN
1,114086,chance-114086-1-0,2,dribbles,NaN,False,24268.0,NaN,24268.0,15347.0,NaN,NaN
2,114086,chance-114086-1-0,3,dribbles,NaN,False,24268.0,NaN,24268.0,15347.0,NaN,NaN
3,114086,chance-114086-1-0,4,dribbles,NaN,False,24268.0,NaN,24268.0,15347.0,NaN,NaN
4,114086,chance-114086-1-0,5,dribbles,NaN,False,24268.0,NaN,24268.0,15347.0,NaN,NaN
5,114086,chance-114086-1-0,6,off_ball_screens,NaN,False,24949.0,59138.0,24268.0,59158.0,NaN,NaN
6,114086,chance-114086-1-0,7,dribbles,NaN,False,24268.0,NaN,24268.0,15347.0,NaN,NaN
7,114086,chance-114086-1-0,8,dribbles,NaN,False,24268.0,NaN,24268.0,15347.0,NaN,NaN
8,114086,chance-114086-1-0,9,dribbles,NaN,False,24268.0,NaN,24268.0,15347.0,NaN,NaN
9,114086,chance-114086-1-0,10,off_ball_screens,NaN,False,24949.0,35096.0,24268.0,59158.0,NaN,NaN


In [206]:
list(actions_df.columns)

['gameId',
 'chanceId',
 'possessionId',
 'startFrame',
 'endFrame',
 'startGameClock',
 'endGameClock',
 'shotClock',
 'offTeamId',
 'defTeamId',
 'playerId',
 'defenderId',
 'closestDefLoc',
 'location',
 'endLoc',
 'numDribbles',
 'ledToShot',
 'touchTime',
 'outcomes',
 'regionsIn',
 'endClosestDefLoc',
 'direct',
 'action_type',
 'action_id',
 'touchId',
 'actor_id',
 'secondary_actor_id',
 'defender_id',
 'receiver_id',
 'gameClock',
 'frame',
 'wallClock',
 'ballhandlerId',
 'region',
 'passerId',
 'receiverId',
 'toReceiverId',
 'complete',
 'turnover',
 'assistOpp',
 'inbounds',
 'backcourt',
 'passerLoc',
 'receiverLoc',
 'passerRegion',
 'receiverRegion',
 'distance',
 'shooterId',
 'three',
 'outcome',
 'fouled',
 'foulId',
 'assisted',
 'blocked',
 'blockerId',
 'dribblesBefore',
 'shotType',
 'complexShotType',
 'catchAndShoot',
 'shotQuality',
 'createdFromPaint',
 'contested',
 'contestLevel',
 'releaseTime',
 'closestDefDist',
 'closestDefId',
 'screenerId',
 'ballhand

In [207]:
actions_df

,gameId,chanceId,possessionId,startFrame,endFrame,startGameClock,endGameClock,shotClock,offTeamId,defTeamId,...,seconds_before_shot,in_lookback,transition,usable,qualityIndex,startType,offPlayerIds,defPlayerIds,sequence_in_chance,chance_has_shot
0,114086,chance-114086-1-0,possession-114086-1-0,510.0,700.0,599.640625,592.053528,24.0,1326,1328,...,NaN,False,False,True,5,JMP,"[24949, 24268, 59136, 59138, 35096]","[59157, 15347, 25182, 59158, 35402]",1,False
1,114086,chance-114086-1-0,possession-114086-1-0,NaN,NaN,NaN,NaN,NaN,1326,1328,...,NaN,False,False,True,5,JMP,"[24949, 24268, 59136, 59138, 35096]","[59157, 15347, 25182, 59158, 35402]",2,False
2,114086,chance-114086-1-0,possession-114086-1-0,NaN,NaN,NaN,NaN,NaN,1326,1328,...,NaN,False,False,True,5,JMP,"[24949, 24268, 59136, 59138, 35096]","[59157, 15347, 25182, 59158, 35402]",3,False
3,114086,chance-114086-1-0,possession-114086-1-0,NaN,NaN,NaN,NaN,NaN,1326,1328,...,NaN,False,False,True,5,JMP,"[24949, 24268, 59136, 59138, 35096]","[59157, 15347, 25182, 59158, 35402]",4,False
4,114086,chance-114086-1-0,possession-114086-1-0,NaN,NaN,NaN,NaN,NaN,1326,1328,...,NaN,False,False,True,5,JMP,"[24949, 24268, 59136, 59138, 35096]","[59157, 15347, 25182, 59158, 35402]",5,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31450,191313,chance-191313-4-203,possession-191313-4-152,140470.0,NaN,4.900000,NaN,NaN,1331,1322,...,NaN,False,False,False,0,FTORB,"[59207, 35222, 58361, 31739, 59205]","[59108, 28410, 59107, 59104, 59110]",1,False
31451,191313,chance-191313-4-204,possession-191313-4-152,142827.0,142885.0,4.000000,2.100000,NaN,1331,1322,...,3.00,True,True,True,4,DEFOB,"[59207, 35222, 58361, 31739, 59205]","[59108, 28410, 59107, 59104, 59110]",1,True
31452,191313,chance-191313-4-204,possession-191313-4-152,NaN,NaN,NaN,NaN,NaN,1331,1322,...,2.20,True,True,True,4,DEFOB,"[59207, 35222, 58361, 31739, 59205]","[59108, 28410, 59107, 59104, 59110]",2,True
31453,191313,chance-191313-4-204,possession-191313-4-152,142885.0,142902.0,2.100000,1.420000,0.0,1331,1322,...,0.68,True,True,True,4,DEFOB,"[59207, 35222, 58361, 31739, 59205]","[59108, 28410, 59107, 59104, 59110]",3,True


In [208]:
# Actions in the chances SkillCorner records a shot for (shots_df), up to and including
# the release; drops the loose-ball / rebound and post-make inbound actions that some
# chances record after the shot.
shot_actions_df = actions_df.loc[
    actions_df['chanceId'].isin(shots_df['chanceId'])
    & (actions_df['frames_before_shot'] >= 0)
]
shot_actions_df.shape

(24148, 133)

In [209]:
actions_df.to_csv(Path.cwd().parent / "data" / "actions.csv", index=False)

In [210]:
shot_actions_df.to_csv(Path.cwd().parent / "data" / "shot_actions.csv", index=False)